# Week 8: First-Order Logic: Forward and Backward Chaining

This week represents first-order rules and facts, then proves queries by forward reasoning. The slides also walk through backward chaining on the same criminality example; both procedures are included.

**Source:** `PPTs/Week8.pptx`.


### Theory note
The crime example uses facts such as `American(Robert)`, `Enemy(A, America)`, `Missile(T1)` and `Owns(A,T1)` with Horn rules. The second exercise reuses this knowledge base to illustrate goal-directed backward chaining. A second forward-chaining example in the deck asks for infection/flu inference from symptoms.

## Program 1 — Forward chaining over Horn clauses

**Theory.** Forward chaining starts with known facts and repeatedly applies rules whose premises are satisfied. The Week 8 deck includes a crime-and-missile example and a fever/sore-throat example. This small engine accepts user-entered Horn rules and facts, then checks whether they derive a query.

**Objective.** Build a first-order Horn-clause knowledge base and determine whether forward reasoning derives a query.

**Libraries.** Python standard library only.

**Algorithm.**
1. Parse facts and rules. 2. Match rule premises against the known facts using substitutions. 3. Add new ground conclusions. 4. Repeat until no new fact is added. 5. Check whether any known fact matches the query.

**Complexity.** Finite, function-free Horn KB: repeated joins can be exponential in rule-body size; storage is O(F), where F is the number of distinct derived facts.

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** The knowledge base derives Criminal(Robert) through the missile, weapon, hostile-country and sale rules.


In [1]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

# Small Horn-clause engine: atoms look like Predicate(term, ...); lowercase names are variables.
def split_top(text,separator=','):
    parts=[]; depth=0; start=0
    for i,ch in enumerate(text):
        if ch=='(': depth+=1
        elif ch==')': depth-=1
        elif ch==separator and depth==0: parts.append(text[start:i].strip()); start=i+1
    parts.append(text[start:].strip()); return [p for p in parts if p]
def parse_atom(text):
    text=text.strip(); i=text.find('(')
    if i<1 or not text.endswith(')'): raise ValueError('Use Predicate(term, ...).')
    name=text[:i].strip(); body=text[i+1:-1].strip()
    return (name,tuple(t.strip() for t in split_top(body)))
def isvar(term): return term and term[0].islower()
def apply_term(term,sub):
    while isvar(term) and term in sub: term=sub[term]
    return term
def unify_atom(pattern,fact,sub=None):
    if pattern[0]!=fact[0] or len(pattern[1])!=len(fact[1]): return None
    out={} if sub is None else dict(sub)
    for p,f in zip(pattern[1],fact[1]):
        p=apply_term(p,out); f=apply_term(f,out)
        if p==f: continue
        if isvar(p): out[p]=f
        elif isvar(f): out[f]=p
        else: return None
    return out
def parse_rule(text):
    if '->' not in text: raise ValueError('Write a rule as Antecedent(x), Condition(x) -> Conclusion(x).')
    left,right=text.split('->',1)
    return ([parse_atom(s) for s in split_top(left)],parse_atom(right))
def render_atom(atom,sub=None):
    sub={} if sub is None else sub
    return atom[0]+'('+', '.join(apply_term(t,sub) for t in atom[1])+')'
def match_premises(premises,facts,index=0,sub=None):
    sub={} if sub is None else sub
    if index==len(premises): yield sub; return
    for fact in facts:
        found=unify_atom(premises[index],fact,sub)
        if found is not None: yield from match_premises(premises,facts,index+1,found)
def forward_chain(facts,rules):
    known=set(facts); rounds=0
    while True:
        additions=set()
        for premises,conclusion in rules:
            for sub in match_premises(premises,known):
                inferred=(conclusion[0],tuple(apply_term(t,sub) for t in conclusion[1]))
                if inferred not in known: additions.add(inferred)
        if not additions: return known,rounds
        known.update(additions); rounds+=1
        print(f'Round {rounds}: '+', '.join(sorted(render_atom(a) for a in additions)))
def main():
    global FACTS,RULES
    try: nr=int(input('Number of Horn rules: '))
    except ValueError: print('Enter an integer.'); return
    if nr<1: print('Enter at least one rule.'); return
    try: RULES=[parse_rule(input(f'Rule {i+1} (premises -> conclusion): ')) for i in range(nr)]
    except ValueError as exc: print('Invalid rule:',exc); return
    facts_text=input('Initial facts separated by semicolons: ')
    try: FACTS=[parse_atom(s) for s in facts_text.split(';') if s.strip()]
    except ValueError as exc: print('Invalid fact:',exc); return
    query_text=input('Query atom: ')
    try: query=parse_atom(query_text)
    except ValueError as exc: print('Invalid query:',exc); return
    known,rounds=forward_chain(FACTS,RULES)
    print(f'\nDerived {len(known)-len(FACTS)} new facts in {rounds} rounds.')
    print('Query proved:',any(unify_atom(query,fact) is not None for fact in known))
    print('Known facts:')
    for atom in sorted(known,key=render_atom): print(' ',render_atom(atom))


SAMPLE_INPUTS = ['4', 'American(x), Weapon(y), Sells(x, y, z), Hostile(z) -> Criminal(x)', 'Missile(x), Owns(A, x) -> Sells(Robert, x, A)', 'Missile(x) -> Weapon(x)', 'Enemy(x, America) -> Hostile(x)', 'Owns(A, T1); Missile(T1); American(Robert); Enemy(A, America)', 'Criminal(Robert)']
with sample_input(SAMPLE_INPUTS):
    main()


Number of Horn rules: 4
Rule 1 (premises -> conclusion): American(x), Weapon(y), Sells(x, y, z), Hostile(z) -> Criminal(x)
Rule 2 (premises -> conclusion): Missile(x), Owns(A, x) -> Sells(Robert, x, A)
Rule 3 (premises -> conclusion): Missile(x) -> Weapon(x)
Rule 4 (premises -> conclusion): Enemy(x, America) -> Hostile(x)
Initial facts separated by semicolons: Owns(A, T1); Missile(T1); American(Robert); Enemy(A, America)
Query atom: Criminal(Robert)
Round 1: Hostile(A), Sells(Robert, T1, A), Weapon(T1)
Round 2: Criminal(Robert)

Derived 4 new facts in 2 rounds.
Query proved: True
Known facts:
  American(Robert)
  Criminal(Robert)
  Enemy(A, America)
  Hostile(A)
  Missile(T1)
  Owns(A, T1)
  Sells(Robert, T1, A)
  Weapon(T1)


## Program 2 — Backward chaining for the same knowledge base

**Theory.** Backward chaining starts at the requested goal and looks for rules that could produce it. It then recursively proves the rule premises from facts or other rules. The Week 8 deck also describes this goal-directed reasoning method for the crime example.

**Objective.** Try to prove a user-entered goal by recursively reducing it to premises.

**Libraries.** Python standard library; reuses the parser, facts and rules from Program 1.

**Algorithm.**
1. Match the goal against known facts. 2. If that fails, find a rule whose conclusion matches the goal. 3. Substitute the matched variables in its premises. 4. Prove every premise recursively while detecting loops.

**Complexity.** Worst case exponential in proof depth; recursion space is O(d), where d is the proof depth.

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** Forward chaining runs first to collect the knowledge base, then the second program searches backward from Criminal(Robert).


In [2]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

def backward_prove(goal,facts,rules,depth=0,trail=None):
    trail=set() if trail is None else set(trail)
    key=render_atom(goal)
    if key in trail or depth>30: return False
    trail.add(key)
    if any(unify_atom(goal,fact) is not None for fact in facts): return True
    for premises,conclusion in rules:
        sub=unify_atom(conclusion,goal)
        if sub is None: continue
        goals=[(a[0],tuple(apply_term(t,sub) for t in a[1])) for a in premises]
        if all(backward_prove(g,facts,rules,depth+1,trail) for g in goals): return True
    return False

def main_backward():
    query_text=input('Goal to prove by backward chaining: ')
    try: query=parse_atom(query_text)
    except ValueError as exc: print('Invalid query:',exc); return
    print('Query proved:',backward_prove(query,FACTS,RULES))

def main():
    query_text=input('Goal to prove by backward chaining: ')
    try: query=parse_atom(query_text)
    except ValueError as exc: print('Invalid query:',exc); return
    print('Query proved:',backward_prove(query,FACTS,RULES))


SAMPLE_INPUTS = ['Criminal(Robert)']
with sample_input(SAMPLE_INPUTS):
    main()


Goal to prove by backward chaining: Criminal(Robert)
Query proved: True
